# The idealised book against the Hawkes book

Section 1.4 of the lecture notes, from Assumption 1.4.1 to Remark 1.4.11.  Under the assumption
the identity (1.4.3) is exact, and Example 1.4.10 places the events of the order flow of
Definition 1.4.6 so that the assumption holds.  The package simulates the same events under the
geometric placement of Definition 1.4.8.  We place one path of events both ways, from one opening
book, and measure how long the two books stay equal and how far apart their mid-prices go once
they part.

Two regimes.  Without withdrawals the two placements can agree for a while, and sections 2 to 5
follow the first discrepancy and the distance after it.  With withdrawals they part almost at once,
and section 6 follows the distance over a long session instead.  Section 7 sets the two side by
side.  The book under the idealised placement carries the superscript $\mathrm{id}$ and the book
under the geometric one $\mathrm{geo}$.  Prices are integer counts of ticks, so $\tau = 1$ in every
formula, and $\eta$ is `DepthDecay`.  Every path is seeded; the fans of sections 3 to 6 take about
two minutes together.

In [ ]:
import dataclasses

import numpy as np
import pandas as pd

from unito26.lob import config
from unito26.lob.coupling import (
    CoupledPlacements, RecordedEvents, agreement_by_case, agreement_probability,
    conditional_survival, distance_terms, first_discrepancy, geometric_mid_moves,
    geometric_readings,
    idealised_readings, limit_event_outcomes, target_offsets, touch_shares, touch_walk_exit,
)
from unito26.lob.hawkes import Clock, ExponentialHawkes
from unito26.lob.idealised import UniformDepth, idealised_book
from unito26.lob.messages import BUY, SELL, GridDepth
from unito26.lob.simulate import EventJournal, EventType, OrderFlowSimulator
from unito26.lob.visualization import (
    MUTED, agreement_figure, coupled_book_figure, coupled_depth_figure, coupled_path_figure,
    discrepancy_survival_panels, distance_terms_figure, fan_panels_figure, use_template,
)

use_template()

FLOW = config.withdrawal_free_order_flow_params()
UNIT = config.unit_mark_params()
UNIFORM_DEPTH = UniformDepth(5)
OPENING = {BUY: (1000, 3), SELL: (1001, 3)}
LEVELS = GridDepth(60)
WARM_UP = 10.0
HORIZON = 20.0
SEED = 11

NAMES = {
    EventType.MARKET_BUY: "market buy",
    EventType.MARKET_SELL: "market sell",
    EventType.LIMIT_BUY: "limit buy",
    EventType.LIMIT_SELL: "limit sell",
    EventType.WITHDRAW_BUY: "withdraw bid",
    EventType.WITHDRAW_SELL: "withdraw ask",
}
PRESSURE = np.array([event.pressure for event in EventType])
LIMIT_TYPES = [EventType.LIMIT_BUY, EventType.LIMIT_SELL]
WITHDRAWALS = [EventType.WITHDRAW_BUY, EventType.WITHDRAW_SELL]


def with_decay(marks, depth_decay):
    """The same marks, with the offsets of (1.4.5) at eta = depth_decay."""
    return dataclasses.replace(marks, depth_decay=depth_decay)


def warmed_path(params, seed, horizon):
    """A path from the empty state, over the warm-up and the horizon after it."""
    return RecordedEvents.drawn(ExponentialHawkes(params, rng=seed), WARM_UP + horizon)


def signed_count(path, clock):
    """varpi' N(t) at every t of clock: the pressure-signed count of the events."""
    return np.r_[0, np.cumsum(PRESSURE[path.types])][
        np.searchsorted(path.times, clock, side="right")]

## 1. One path of events, two placements

The order flow is a marked point process: the Hawkes process gives the times and the types, and
the marks, the sizes and the offsets $\zeta_n$ of (1.4.5), are drawn independently of both.  To
compare two placements the events must be the same and only the placement may differ.  The
simulator draws its events and its marks on one generator while the idealised placement draws no
marks, so a Hawkes process and a simulator on the same seed give different paths: every mark drawn
shifts the draws that follow.

In [ ]:
def first_events(types, times, count):
    return pd.DataFrame({
        "time (s)": np.round(times[:count], 4),
        "event": [NAMES[EventType(event)] for event in types[:count]],
    })


hawkes = RecordedEvents.drawn(ExponentialHawkes(FLOW, rng=SEED), 5.0)

simulator = OrderFlowSimulator(FLOW, UNIT, OPENING[BUY][0], rng=SEED)
book, journal = idealised_book(UNIFORM_DEPTH, OPENING, LEVELS), EventJournal.empty()
for message in simulator.stream(book, 5.0, journal):
    book.apply(message, record=False)

pd.concat({
    "ExponentialHawkes, seed 11": first_events(hawkes.types, hawkes.times, 5),
    "OrderFlowSimulator, seed 11": first_events(journal.types, np.array(journal.times), 5),
}, axis=1)

The first event is common to both, and the second is not: the simulator has drawn a size in
between.  So the events are drawn once, as a `RecordedEvents`, and each placement reads its own
replay of them: `IdealisedFlow` directly, and the simulator through
`OrderFlowSimulator.driven_by`, which takes the events from outside and draws only the marks.
`CoupledPlacements.on` does both from copies of one opening book, and records the two sessions
event by event.

The opening book is `idealised_book` with $\bar S = 5$, three shares at each best quote and a
spread of one tick, sixty levels a side.  The path is read after a warm-up of ten seconds from the
empty state, twenty relaxation times $1/(\beta(1-\rho))$, so that the flow has forgotten its empty
past.

In [ ]:
events = warmed_path(FLOW, SEED, HORIZON).after(WARM_UP)
pair = CoupledPlacements.on(events, UNIT, UNIFORM_DEPTH, OPENING, LEVELS, SEED)

print(f"relaxation time {FLOW.relaxation_time():.2f} s; {len(events)} events on "
      f"(0, {HORIZON:g}] s after the warm-up")
print("both placements read every event, in order:",
      all(np.array_equal(j.types, events.types)
          for j in (pair.idealised_journal, pair.geometric_journal)))
print("the idealised placement reached the horizon:", pair.breach is None)
print(f"the books first differ after event {pair.first_discrepancy}, "
      f"at {pair.first_discrepancy_time:.3f} s")

## 2. Without withdrawals: the closest case

`withdrawal_free_order_flow_params()` is the resilient specification with the withdrawals switched
off, limit orders arriving on each side at the rate of the market orders that consume it.
`unit_mark_params()` has orders of one share and $\eta = 1$.  A market order is then the same order
under both placements, one share at the opposite best quote.  A limit order is priced by (1.4.6)
at $1 + \zeta_n$ ticks from the opposite best quote, and by Example 1.4.10 at the best quote of its
side, or one tick beyond it once that quote holds $\bar S$.  The two are the same order exactly
when $\zeta_n = \zeta^*_n$, where, on the book the event finds,

$$
\zeta^*_n = \frac{\phi_{T_n-}}{\tau} - 1 - \mathbf 1\{S^d_{T_n-} = \bar S\},
$$

$S^d$ being the size at the best quote of the order's side.  Case by case, at a spread of
$k$ ticks:

| book before | idealised limit order | the same order iff | probability, by (1.4.5) |
| --- | --- | --- | --- |
| $S^d < \bar S$ | joins its best quote | $\zeta_n = k - 1$ | $\eta(1-\eta)^{k-1}$ |
| $S^d = \bar S$, $k \ge 2$ | one tick inside the spread | $\zeta_n = k - 2$ | $\eta(1-\eta)^{k-2}$ |
| $S^d = \bar S$, $k = 1$ | trades a share at the opposite best quote | never | $0$ |

So the probability of agreement is $\eta(1-\eta)^{\zeta^*_n}$, and zero at $\zeta^*_n = -1$.  Let
$\chi^{\neq}$ be the first event time after which the books differ.  Up to it the geometric book
*is* the idealised book, so the rule decides every event up to $\chi^{\neq}$, read on the idealised
book.  At $\eta = 1$ every offset is zero, and the books part at the first limit event with
$\zeta^*_n \neq 0$.

The two books on the path of section 1, up to a few seconds after they part.  The geometric book is
solid and the idealised one dotted, so that up to the dashed rule the second lies under the first.
The dotted grey lines in the lower panels are $\bar S$.

In [ ]:
parting = pair.first_discrepancy_time
figure = coupled_path_figure(pair, title="The two books on one path of events, η = 1")
for row in (2, 3):
    figure.add_hline(y=UNIFORM_DEPTH, line=dict(color=MUTED, width=1, dash="dot"), row=row, col=1)
figure.update_xaxes(range=[0.0, parting + 4.0])
figure

The event the books part at, and the best quotes either side of it:

In [ ]:
first = pair.first_discrepancy
offsets = target_offsets(events, UNIFORM_DEPTH, OPENING, LEVELS)
case = offsets.loc[first]
print(f"event {first}, a {NAMES[EventType(case.EventType)]} at {case.TimeStamp:.3f} s: spread "
      f"{case.Spread} tick, its own best quote at Sbar: {case.TouchFull}, "
      f"target offset {case.TargetOffset}")

COLUMNS = ["MidPrice", "BidPrice1", "BidSize1", "AskPrice1", "AskSize1"]
pd.DataFrame({
    (placement, when): pair.frame.loc[row, [f"{placement}{column}" for column in COLUMNS]].to_numpy()
    for placement in ("Idealised", "Geometric")
    for when, row in (("before", first - 1), ("after", first))
}, index=COLUMNS)

It is the third case of the table.  The best bid holds $\bar S$ at a spread of one tick, so the
idealised limit buy trades a share at the best ask, while the geometric one, at $\zeta_n = 0$, rests
one tick below the best ask, which is the best bid: the bid holds $\bar S + 1$, against part
*iii.* of the assumption.  The mid-prices are still equal after it, and part at a later event,
when one book clears a queue that the other does not.  The first discrepancy is a lower bound for
the first time the mids differ.

The two books after the event, side by side:

In [ ]:
coupled_book_figure(pair, first, 4, title=f"The two books after event {first}")

And the books around the best quotes over the same stretch of time, the bid in blue and the ask in
orange, the size of a queue in the depth of its colour:

In [ ]:
mid = pair.frame.loc[first, "IdealisedMidPrice"]
figure = coupled_depth_figure(pair)
figure.update_xaxes(range=[parting - 1.0, parting + 4.0])
figure.update_yaxes(range=[mid - 6, mid + 6], dtick=1)
figure

## 3. The rule, event by event

A fan of 1000 paths of the withdrawal-free flow, each over twenty seconds after its warm-up, folded
into the opening book of section 1 with orders of one share at three values of $\eta$.  At every
limit event up to and including the first discrepancy the book is the idealised one, so the event
is a trial of the rule: its case is read on the idealised book, and it agrees or it does not.  The
trials are grouped by the spread and by whether the best quote of the order's side held $\bar S$.

In [ ]:
PATHS = 1000
warmed = [warmed_path(FLOW, 1000 + m, HORIZON) for m in range(PATHS)]
fan = [path.after(WARM_UP) for path in warmed]
states = np.array([path.state_after(WARM_UP) for path in warmed])

BOOKS = {
    "S̄ = 5": (UNIFORM_DEPTH, OPENING),
    "S̄ = 50": (UniformDepth(50), {BUY: (1000, 25), SELL: (1001, 25)}),
}
fan_offsets = {
    name: [target_offsets(path, depth, opening, LEVELS) for path in fan]
    for name, (depth, opening) in BOOKS.items()
}


def first_discrepancies(name, depth_decay):
    """The first discrepancy event of every path of the fan, from the opening book `name`."""
    depth, opening = BOOKS[name]
    marks = with_decay(UNIT, depth_decay)
    return [first_discrepancy(path, marks, depth, opening, LEVELS, m)
            for m, path in enumerate(fan)]


RULE_DECAYS = (0.8, 0.5, 0.2)
firsts = {("S̄ = 5", decay): first_discrepancies("S̄ = 5", decay) for decay in RULE_DECAYS}
tables = {
    decay: agreement_by_case(pd.concat([
        limit_event_outcomes(offsets, first)
        for offsets, first in zip(fan_offsets["S̄ = 5"], firsts["S̄ = 5", decay])
    ]), decay)
    for decay in RULE_DECAYS
}
deviations = pd.concat({
    decay: (table["Agreed"] - table["Formula"]) / table["StandardError"].where(
        table["StandardError"] > 0)
    for decay, table in tables.items()
})
print("cases with at least 30 trials, largest |frequency - formula| in standard errors:",
      round(deviations[pd.concat(tables)["LimitEvents"] >= 30].abs().max(), 2))
tables[0.5].round(4)

The table is $\eta = 0.5$.  Every case observed often enough meets its formula within the
deviation printed above, at each $\eta$.  The figure sets the three side by side, for the cases
observed at least twenty times: a case is a marker at its target offset, a circle when the best
quote held less than $\bar S$ and a diamond when it was full, with two standard errors either side.  The cases at one offset share a probability, and the
full quote facing a spread of one tick, at $\zeta^* = -1$, never agrees.

In [ ]:
agreement_figure({decay: table[table["LimitEvents"] >= 20] for decay, table in tables.items()},
                 title="Agreement at a limit event against η(1−η)<sup>ζ*</sup>, 1000 paths, S̄ = 5")

## 4. The first discrepancy

The offsets are independent of the events, and $\zeta^*_n$ is read on the idealised book, which
is a function of the events alone.  So, given the events, the books survive to $t$ with probability

$$
\mathbb P(\chi^{\neq} > t \mid \text{the events})
= \prod_{n \,:\, T_n \le t,\ E_n \in \mathrm{lim}} \eta(1-\eta)^{\zeta^*_n}
= \eta^{N_{\mathrm{lim}}(t)}\,(1-\eta)^{J(t)},
$$

with $N_{\mathrm{lim}}$ the count of limit orders and $J(t)$ the sum of their target offsets,
infinite if one of them is $-1$.  Averaged over the paths, the product is
$\mathbb P(\chi^{\neq} > t)$ computed without the noise of the offsets.  It is at most
$\mathbb E[\eta^{N_{\mathrm{lim}}(t)}]$, which the exponential kernel gives in closed form from the
state $z = Z(0+)$: with $x = \eta$ on the limit types and one elsewhere,

$$
\mathbb E\Big[\prod_e x_e^{N_e(t)} \,\Big|\, Z(0+) = z\Big]
= \exp\Big(\psi(t)^\top z + \int_0^t \mu^\top\big(x\circ e^{\psi(r)} - \mathbf 1\big)\,dr\Big),
\qquad
\psi' = -\beta\psi + A^\top\big(x\circ e^{\psi} - \mathbf 1\big),\ \psi(0) = 0,
$$

which `HawkesParams.count_generating_function` solves.  The bound is the value when every target
offset is zero.  That holds from a spread of one tick until a best quote empties or a limit order
meets a full one: the first time $\chi^{\partial}$ that the size $\Upsilon^b$ or $\Upsilon^a$ at a
best quote, which moves by one share at each event acting on it, leaves $[1, \bar S]$.  At
$\eta = 1$ nothing parts the books before it, so $\mathbb P(\chi^{\neq} > t) \ge
\mathbb P(\chi^{\partial} > t)$ there.

We fold the fan from two opening books: the book of section 1, with $\bar S = 5$, and one with
$\bar S = 50$ and 25 shares at each best quote, where $\chi^{\partial}$ lies far beyond the horizon
and the bound is the value.  The Riccati value is averaged over the states the paths start from.

In [ ]:
DECAYS = (1.0, 0.5, 0.2, 0.08)
CLOCK = Clock(np.geomspace(0.02, HORIZON, 400))

for name in BOOKS:
    for decay in DECAYS:
        if (name, decay) not in firsts:
            firsts[name, decay] = first_discrepancies(name, decay)
walk_exits = {
    name: np.array([touch_walk_exit(path, opening, depth) or np.inf for path in fan])
    for name, (depth, opening) in BOOKS.items()
}


def discrepancy_times(name, decay):
    return np.array([path.times[n] if n is not None else np.nan
                     for path, n in zip(fan, firsts[name, decay])])


def limit_weights(decay):
    weights = np.ones(len(EventType))
    weights[LIMIT_TYPES] = decay
    return weights


times, curves, summary = {}, {}, []
for name in BOOKS:
    times[name], curves[name] = {}, {}
    for decay in DECAYS:
        label = f"η = {decay:g}"
        sample = discrepancy_times(name, decay)
        averaged = np.mean([
            conditional_survival(offsets["TimeStamp"],
                                 agreement_probability(offsets["TargetOffset"], decay), CLOCK)
            for offsets in fan_offsets[name]
        ], axis=0)
        bound = FLOW.count_generating_function(limit_weights(decay), states, CLOCK).mean(axis=0)
        times[name][label] = sample
        curves[name][label] = {"the conditional product, averaged": averaged,
                               "E[η<sup>N<sub>lim</sub>(t)</sup>], the Riccati system": bound}
        if decay == 1.0:
            curves[name][label]["P(χ<sup>∂</sup> > t)"] = (
                walk_exits[name][:, None] > CLOCK).mean(axis=0)
        survived = ~(sample[:, None] <= CLOCK)
        empirical = survived.mean(axis=0)
        error = np.sqrt(np.maximum(averaged * (1 - averaged), 1 / PATHS) / PATHS)
        summary.append({
            "book": name, "η": decay,
            "median χ≠ (s)": np.median(np.where(np.isnan(sample), np.inf, sample)),
            "P(χ≠ > 1 s)": np.mean(~(sample <= 1.0)),
            "max |empirical - product|": np.abs(empirical - averaged).max(),
            "largest, in standard errors": (np.abs(empirical - averaged) / error).max(),
            "max |product - Riccati|": np.abs(averaged - bound).max(),
        })

pd.DataFrame(summary).set_index(["book", "η"]).round(3)

In [ ]:
discrepancy_survival_panels(
    times, CLOCK, curves,
    title="P(χ<sup>≠</sup> > t): the coupled runs against the formulas, 1000 paths",
)

The empirical survival follows the averaged product within two standard errors at every time, in
every panel.  At $\bar S = 50$ the Riccati value is the answer, to within the noise of a thousand
paths: no best quote fills or empties in twenty seconds, every target offset is zero, and the
product is $\eta^{N_{\mathrm{lim}}(t)}$ itself.  At $\bar S = 5$ it is close to the answer at small $\eta$,
where the books part at one of the first limit orders, before the spread has left one tick.  It is
no bound at all at $\eta = 1$, where it is one and the books part all the same, at the first limit
event after $\chi^{\partial}$ that is not one tick inside a full best quote.

Which case ends the agreement at $\eta = 1$, from the book with $\bar S = 5$:

In [ ]:
ending = pd.DataFrame([
    offsets.loc[n, ["Spread", "TouchFull", "TargetOffset"]]
    for offsets, n in zip(fan_offsets["S̄ = 5"], firsts["S̄ = 5", 1.0]) if n is not None
])
ending.value_counts(normalize=True).rename("share of the paths").to_frame().round(3)

## 5. After they part

The identity (1.4.1) relates the order flow imbalance to the best quotes on any book.  Telescoped
over the events of $(0, t]$ it reads

$$
\mathrm{OFI}_{t,t} - \big(S^b_t - S^b_0\big) + \big(S^a_t - S^a_0\big)
= \sum_{0 < T_n \le t} \Big[\operatorname{sgn}(\Delta P^b_n)\,\hat S^b_n
+ \operatorname{sgn}(\Delta P^a_n)\,\hat S^a_n\Big],
$$

where, when a best price moves, $\hat S$ is the size at the outer of its two values (the lower of
the two best bids, the higher of the two best asks), read on the book in which that price is best.
On the idealised book every move is one tick onto or off a queue of $\bar S$, so the right side is
$2\bar S\,(P^m_t - P^m_0)$, and the identity is (1.4.3).  On the geometric book a move is $k$ ticks
onto or off a queue of any size.  Subtracting the two,

$$
2\bar S\big(P^{m,\mathrm{geo}}_t - P^{m,\mathrm{id}}_t\big)
= \underbrace{\mathrm{OFI}^{\mathrm{geo}}_{t,t} - \mathrm{OFI}^{\mathrm{id}}_{t,t}}_{(a)}
- \underbrace{\Big[\big(S^{b,\mathrm{geo}}_t - S^{b,\mathrm{id}}_t\big)
- \big(S^{a,\mathrm{geo}}_t - S^{a,\mathrm{id}}_t\big)\Big]}_{(b)}
- \underbrace{\sum_{0<T_n\le t}\Big[\operatorname{sgn}(\Delta P^{b,\mathrm{geo}}_n)
\big(\hat S^{b,\mathrm{geo}}_n - \bar S k^b_n\big)
+ \operatorname{sgn}(\Delta P^{a,\mathrm{geo}}_n)\big(\hat S^{a,\mathrm{geo}}_n - \bar S k^a_n\big)
\Big]}_{(c)},
$$

exactly, while both sides of both books are occupied.  Term (a) is the order flow the geometric
book does not see, the limit orders that rest behind its best quotes; (b) is the residual
(1.4.4) of each book; (c) is the depth per tick of the geometric moves against $\bar S$.  On one
path of two minutes, with lognormal sizes and $\eta = 0.5$:

In [ ]:
LOGNORMAL = dataclasses.replace(UNIT, sigma_log_size=config.resilient_mark_params().sigma_log_size)

path = warmed_path(FLOW, SEED, 120.0).after(WARM_UP)
long_pair = CoupledPlacements.on(path, with_decay(LOGNORMAL, 0.5), UNIFORM_DEPTH, OPENING,
                                 LEVELS, SEED)
terms = distance_terms(long_pair, UNIFORM_DEPTH, OPENING)
print("the three terms add up to the distance at every event:",
      np.array_equal(terms["Distance"], terms["ImbalanceMismatch"]
                     - terms["TouchSizeMismatch"] - terms["DepthMismatch"]))
print("the sum (c) on the idealised book is zero at every event:",
      (terms["IdealisedDepthMismatch"] == 0).all())
distance_terms_figure(terms, title="The distance and its three terms, lognormal sizes, η = 0.5")

The lognormal sizes have the dispersion of `resilient_mark_params()`, $\sigma = 0.8$, about a
median of one share.  Terms (a) and (c) are large and largely cancel: the orders the geometric book
places behind its quotes are missing from its imbalance, and are also the depth its later moves
travel through.

Two properties hold for the law of the distance.  The specification is direction-symmetric and the
opening book symmetric about its mid, so exchanging every type with its mirror reflects both books
and the distance has a law symmetric about zero.  And with $\gamma_t$ the slope of
$P^{m,\mathrm{geo}}_t$ on the signed count $\varpi^\top N(t)$ and $\xi_t$ what it leaves,

$$
\gamma_t = \frac{\mathrm{Cov}\big(P^{m,\mathrm{geo}}_t, \varpi^\top N(t)\big)}
{\mathrm{Var}\big(\varpi^\top N(t)\big)},
\qquad
\xi_t = P^{m,\mathrm{geo}}_t - P^m_0 - \gamma_t\,\varpi^\top N(t),
$$

the variance of the distance is
$Q_t = (\gamma_t - 1/(2\bar S))^2\,\mathrm{Var}(\varpi^\top N(t)) + \mathrm{Var}(\xi_t)$ to within
$1 + 2\sqrt{Q_t}$, since with unit sizes on the idealised book
$P^{m,\mathrm{id}}_t - P^m_0 = \varpi^\top N(t)/(2\bar S)$ plus a residual of at most a tick.  The
idealised book moves by $1/(2\bar S)$ a unit of signed count; $\gamma_t$ is what the geometric book
moves by.

A fan of 60 paths over five minutes, at two size laws and three values of $\eta$:

In [ ]:
LONG = 300.0
FAN_PATHS = 60
READ = Clock(np.arange(0.0, LONG + 1.0, 5.0))
SIZE_LAWS = {"unit sizes": UNIT, "lognormal sizes": LOGNORMAL}
DISTANCE_DECAYS = (1.0, 0.5, 0.08)

long_paths = [warmed_path(FLOW, 7000 + m, LONG).after(WARM_UP) for m in range(FAN_PATHS)]
counts = np.array([signed_count(path, READ) for path in long_paths])
idealised = np.array([idealised_readings(path, UNIFORM_DEPTH, OPENING, LEVELS, READ)["MidPrice"]
                      for path in long_paths])
geometric = {
    (law, decay): [geometric_readings(path, with_decay(marks, decay), UNIFORM_DEPTH, OPENING,
                                      LEVELS, READ, m)
                   for m, path in enumerate(long_paths)]
    for law, marks in SIZE_LAWS.items() for decay in DISTANCE_DECAYS
}
geometric_mid = {key: np.array([r["MidPrice"] for r in runs]) for key, runs in geometric.items()}


def second_moments(mid, count, idealised_mid, k):
    """The statistics of the second-moment statement at the k-th time of READ."""
    opening = 0.5 * (OPENING[BUY][0] + OPENING[SELL][0])
    move, signed, distance = mid[:, k] - opening, count[:, k], mid[:, k] - idealised_mid[:, k]
    slope = np.cov(move, signed)[0, 1] / signed.var(ddof=1)
    remainder = move - slope * signed
    q = (slope - 1 / (2 * UNIFORM_DEPTH)) ** 2 * signed.var(ddof=1) + remainder.var(ddof=1)
    t = READ[k]
    return {
        "mean distance": distance.mean(),
        "its standard error": distance.std(ddof=1) / np.sqrt(distance.size),
        "Var(distance) / t": distance.var(ddof=1) / t,
        "2 Sbar gamma_t": 2 * UNIFORM_DEPTH * slope,
        "Var(xi_t) / t": remainder.var(ddof=1) / t,
        "|Var(distance) - Q_t| / (1 + 2 sqrt Q_t)":
            abs(distance.var(ddof=1) - q) / (1 + 2 * np.sqrt(q)),
    }


rows = {
    (law, decay, int(READ[k])): second_moments(mid, counts, idealised, k)
    for (law, decay), mid in geometric_mid.items() for k in (12, 30, 60)
}
moments = pd.DataFrame(rows).T.rename_axis(["sizes", "η", "t (s)"])
D = FLOW.count_covariance_rate()
print(f"varpi' D varpi / (2 Sbar)^2 = {PRESSURE @ D @ PRESSURE / (2 * UNIFORM_DEPTH) ** 2:.4f}; "
      "Var(P^m,id_t - P^m_0) / t at 60, 150, 300 s:",
      np.round([(idealised[:, k] - 1000.5).var(ddof=1) / READ[k] for k in (12, 30, 60)], 4))
moments.round(3)

In [ ]:
def root_mean_square(values):
    return np.sqrt(np.mean(values ** 2, axis=0))


distances = {
    law: {f"η = {decay:g}": np.abs(geometric_mid[law, decay] - idealised)
          for decay in DISTANCE_DECAYS}
    for law in SIZE_LAWS
}
summaries = {
    law: {
        f"η = {decay:g}": {
            "√E[(P<sup>m,geo</sup> − P<sup>m,id</sup>)²]": root_mean_square(
                geometric_mid[law, decay] - idealised),
            "√E[(P<sup>m,id</sup> − P<sup>m</sup><sub>0</sub>)²]": root_mean_square(
                idealised - 1000.5),
        }
        for decay in DISTANCE_DECAYS
    }
    for law in SIZE_LAWS
}
fan_panels_figure(distances, READ, summaries, 20, "ticks", "all",
                  title="|P<sup>m,geo</sup> − P<sup>m,id</sup>| over a fan of 60 paths")

The mean distance lies within two standard errors of zero throughout, as the symmetry requires, and
the second-moment statement holds with room to spare.  With unit sizes at $\eta = 1$ the distance
spreads about as fast as the idealised mid itself and $2\bar S\gamma_t$ is near one: the two mids
respond alike to the signed count, and differ by what each adds to it.  As $\eta$ falls, and with
lognormal sizes, the geometric mid responds more to the signed count than the idealised one,
$2\bar S\gamma_t > 1$, and moves much more besides: at $\eta = 0.08$ the distance spreads an order
of magnitude faster.  The slope $\gamma_t$ also drifts with $t$, upward at small $\eta$.  There is
no constant $\gamma(\eta)$ to estimate here: the depth of the geometric book is a walk (section 6),
the book has no stationary state, and the table describes a growth over a stated horizon, not a
rate.

## 6. With withdrawals

`resilient_order_flow_params()` switches the withdrawals on.  Under the geometric placement a
withdrawal takes its size from a level drawn in proportion to the size resting there, over the
whole side (Definition 1.4.8 *iii.*); under the idealised one it takes a share from the best quote.
With orders of one share the two agree exactly when the level drawn is the best quote, which on the
idealised book has probability $S^d/S^d(\tau\mathbb N)$, the share of its side that rests at the
best quote.  That is one in a hundred on the opening book.  The rule of section 2 extends: given
the events, the books survive to $t$ with the product of $\eta(1-\eta)^{\zeta^*_n}$ over the limit
orders and of $S^d/S^d(\tau\mathbb N)$ over the withdrawals.  Over 400 paths of two seconds:

In [ ]:
RESILIENT = config.resilient_order_flow_params()
SHORT = 2.0
SHORT_CLOCK = Clock(np.geomspace(0.002, SHORT, 300))
short_fan = [warmed_path(RESILIENT, 20000 + m, SHORT).after(WARM_UP) for m in range(400)]
short_cases = [(target_offsets(path, UNIFORM_DEPTH, OPENING, LEVELS),
                touch_shares(path, UNIFORM_DEPTH, OPENING, LEVELS)) for path in short_fan]


def survival_with_withdrawals(offsets, shares, decay, clock):
    """The conditional product over the limit orders and the withdrawals of one path."""
    times = np.r_[offsets["TimeStamp"], shares["TimeStamp"]]
    agreement = np.r_[agreement_probability(offsets["TargetOffset"], decay), shares["TouchShare"]]
    order = np.argsort(times)
    return conditional_survival(times[order], agreement[order], clock)


short_times, short_curves, short_summary = {"withdrawals on": {}}, {"withdrawals on": {}}, []
for decay in (1.0, 0.2, 0.08):
    label = f"η = {decay:g}"
    marks = with_decay(UNIT, decay)
    parted = [first_discrepancy(path, marks, UNIFORM_DEPTH, OPENING, LEVELS, m)
              for m, path in enumerate(short_fan)]
    sample = np.array([path.times[n] if n is not None else np.nan
                       for path, n in zip(short_fan, parted)])
    short_times["withdrawals on"][label] = sample
    short_curves["withdrawals on"][label] = {
        "the conditional product, averaged": np.mean(
            [survival_with_withdrawals(o, w, decay, SHORT_CLOCK) for o, w in short_cases], axis=0)
    }
    short_summary.append({"η": decay, "median χ≠ (s)": np.nanmedian(sample),
                          "P(χ≠ > 0.1 s)": np.mean(~(sample <= 0.1))})
print("share of the best quote at the first withdrawal, averaged over the paths:",
      round(np.mean([shares["TouchShare"].iloc[0] for _, shares in short_cases]), 4))
print(pd.DataFrame(short_summary).set_index("η").round(3).to_string())
discrepancy_survival_panels(
    short_times, SHORT_CLOCK, short_curves,
    title="P(χ<sup>≠</sup> > t) with withdrawals, unit sizes, 400 paths",
)

The books part within a tenth of a second on most paths, at the first withdrawal or at a limit
order before it, and the averaged product follows the coupled runs.  With sizes other than one the
first order already differs: a market order of $q_n$ shares against one of a share.

So there is no first discrepancy to speak of, and the question is the distance over a long session.
That needs a geometric book whose depth is stationary, and with orders of one share it is not.
Each side receives the limit orders of its direction and loses the market orders of the other and
its own withdrawals, at the stationary rates:

In [ ]:
rates = pd.Series(RESILIENT.stationary_intensity(), index=list(NAMES.values()))
print(rates.round(3).to_string())
print(f"per side, with orders of one share: in {rates['limit buy']:.3f} shares/s, out "
      f"{rates['market sell'] + rates['withdraw bid']:.3f} shares/s")

The flow out exceeds the flow in, and the geometric book drains.  The specification was calibrated
with the sizes of `resilient_mark_params()`, under which a withdrawal takes at most what rests at
its level.  We use those sizes counted in lots of ten shares, which is the same law: a size of $q$
shares is $q/10$ lots, the book scales with the sizes, and the idealised order of one share becomes
an order of one lot.  The idealised book still counts its orders in units, so it loses about a
quarter of a share a second from each side, and its spread widens by tens of ticks over the session;
(1.4.3) holds all the same.  It opens with 200 levels a side, so that it does not empty.

Thirty paths of seven and a half minutes, at $\eta = 0.08$, the value of
`resilient_mark_params()`, and at $\eta = 0.2$.  The depth of the geometric book, against that of
the two other settings:

In [ ]:
LOTS = dataclasses.replace(config.resilient_mark_params(), mean_log_size=4.0 - np.log(10.0),
                           lot=1)
WIDE = GridDepth(200)
SESSION = 450.0
SESSION_PATHS = 30
SESSION_READ = Clock(np.arange(0.0, SESSION + 1.0, 5.0))
SESSION_DECAYS = (0.08, 0.2)

session_paths = [warmed_path(RESILIENT, 30000 + m, SESSION).after(WARM_UP)
                 for m in range(SESSION_PATHS)]
session_counts = np.array([signed_count(path, SESSION_READ) for path in session_paths])
session_idealised_readings = [
    idealised_readings(path, UNIFORM_DEPTH, OPENING, WIDE, SESSION_READ)
    for path in session_paths
]
session_idealised = np.array([r["MidPrice"] for r in session_idealised_readings])
session = {
    decay: [geometric_readings(path, with_decay(LOTS, decay), UNIFORM_DEPTH, OPENING, WIDE,
                               SESSION_READ, m)
            for m, path in enumerate(session_paths)]
    for decay in SESSION_DECAYS
}
session_mid = {decay: np.array([r["MidPrice"] for r in runs]) for decay, runs in session.items()}
print("an empty side in the geometric book at a reading:",
      any(np.isnan(mid).any() for mid in session_mid.values()))

DEPTH_PATHS = 10
depth_settings = {
    "no withdrawals, unit sizes, η = 0.5": [
        geometric_readings(warmed_path(FLOW, 30000 + m, SESSION).after(WARM_UP),
                           with_decay(UNIT, 0.5), UNIFORM_DEPTH, OPENING, LEVELS, SESSION_READ, m)
        for m in range(DEPTH_PATHS)],
    "withdrawals, unit sizes, η = 0.08": [
        geometric_readings(path, with_decay(UNIT, 0.08), UNIFORM_DEPTH, OPENING, LEVELS,
                           SESSION_READ, m)
        for m, path in enumerate(session_paths[:DEPTH_PATHS])],
    "withdrawals, sizes in lots, η = 0.08": session[0.08][:DEPTH_PATHS],
}
depths = {"bid": {name: np.array([r["BidDepth"] for r in runs])
                  for name, runs in depth_settings.items()}}
depth_curves = {"bid": {name: {"mean over the paths": values.mean(axis=0)}
                        for name, values in depths["bid"].items()}}
fan_panels_figure(depths, SESSION_READ, depth_curves, DEPTH_PATHS, "shares (lots)", False,
                  title="The total size on the bid of the geometric book, S<sup>b</sup>(τℕ)")

Without withdrawals the depth is a walk with no drift, since limit orders and market orders arrive
at one rate, and its spread across paths grows with time.  With withdrawals and unit sizes it
drains.  With the sizes the specification was calibrated for it rises from the opening book and
settles after about two minutes, and we read the session from 150 s on.

The idealised book is not stationary in this regime, but its mid does not inherit the drift: by
(1.4.3), with orders of one share, $P^{m,\mathrm{id}}_t - P^m_0 = \varpi^\top N(t)/(2\bar S)$ plus a
residual under a tick, whatever its spread does.

In [ ]:
residual = session_idealised - 0.5 * (OPENING[BUY][0] + OPENING[SELL][0]) \
    - session_counts / (2 * UNIFORM_DEPTH)
idealised_depth = np.array([r["BidDepth"] for r in session_idealised_readings])
print(f"largest |P^m,id_t - P^m_0 - varpi'N(t) / (2 Sbar)| over every reading: "
      f"{np.abs(residual).max():.2f} ticks")
print(f"total size on the idealised bid: {idealised_depth[:, 0].mean():.0f} at the opening, "
      f"{idealised_depth[:, -1].mean():.0f} after {SESSION:.0f} s, on average over the paths")

So whatever rate the distance grows at is a property of the geometric book alone.  Suppose that
book stationary, and that no side of it empties.  Then the change of its mid over a window splits
into a martingale and a bounded part, and so does the signed count, and over a window of length $h$

$$
\mathrm{Var}(\Delta\,\text{distance}) = c + r\,h + o(1),
\qquad
r = \Big(\gamma - \frac{1}{2\bar S}\Big)^2 \varpi^\top D\varpi + \sigma_\perp^2 ,
$$

where $\gamma$ is the limit slope of the geometric mid on the signed count, $\sigma_\perp^2$ the rate
of the part $\xi$ orthogonal to it, positive as soon as the marks change the moves of the mid, and
$c$ comes from the bounded part, a bounce of the geometric mid that does not accumulate.  Every
second moment over a window has the same affine form, so slopes in $h$ estimate the rates: $\gamma$
is the slope of $\mathrm{Cov}(\Delta P^{m,\mathrm{geo}}, \Delta\varpi^\top N)$ over that of
$\mathrm{Var}(\Delta\varpi^\top N)$, and $\sigma_\perp^2$ the slope of $\mathrm{Var}(\Delta\xi)$.

In [ ]:
SETTLED = 150.0
WINDOWS = (10, 25, 50, 100, 150)


def window_changes(values, h):
    """The changes of each path over the non-overlapping windows of length h after SETTLED."""
    step = int(round(h / 5.0))
    starts = np.arange(int(SETTLED / 5.0), SESSION_READ.size - step, step)
    return (values[:, starts + step] - values[:, starts]).ravel()


def window_moments(mid, h):
    """The second moments over the windows of length h after SETTLED."""
    move, signed = window_changes(mid, h), window_changes(session_counts, h)
    return {
        "windows": move.size,
        "Var(Δ distance)": window_changes(mid - session_idealised, h).var(ddof=1),
        "Cov(Δ P^m,geo, Δ varpi'N)": np.cov(move, signed)[0, 1],
        "Var(Δ varpi'N)": signed.var(ddof=1),
        "Var(Δ P^m,id)": window_changes(session_idealised, h).var(ddof=1),
    }


def slope(values):
    return np.polyfit(WINDOWS, values, 1)[0]


signed_rate = PRESSURE @ RESILIENT.count_covariance_rate() @ PRESSURE
windows = pd.DataFrame({
    (decay, h): window_moments(session_mid[decay], h)
    for decay in SESSION_DECAYS for h in WINDOWS
}).T.rename_axis(["η", "h (s)"])

rates = []
for decay in SESSION_DECAYS:
    table = windows.loc[decay]
    gamma = slope(table["Cov(Δ P^m,geo, Δ varpi'N)"]) / slope(table["Var(Δ varpi'N)"])
    orthogonal = slope([
        (window_changes(session_mid[decay], h)
         - gamma * window_changes(session_counts, h)).var(ddof=1)
        for h in WINDOWS
    ])
    rate, constant = np.polyfit(WINDOWS, table["Var(Δ distance)"], 1)
    rates.append({
        "η": decay, "c": constant, "r": rate, "2 Sbar gamma": 2 * UNIFORM_DEPTH * gamma,
        "sigma_perp^2": orthogonal,
        "(gamma - 1/(2 Sbar))^2 varpi'D varpi + sigma_perp^2":
            (gamma - 1 / (2 * UNIFORM_DEPTH)) ** 2 * signed_rate + orthogonal,
    })
rates = pd.DataFrame(rates).set_index("η")
signed_slope = slope(windows.loc[SESSION_DECAYS[0], "Var(Δ varpi'N)"])
idealised_slope = slope(windows.loc[SESSION_DECAYS[0], "Var(Δ P^m,id)"])
print(f"slope of Var(Δ varpi'N) in h {signed_slope:.3f}, against varpi' D varpi = "
      f"{signed_rate:.3f}; slope of Var(Δ P^m,id) {idealised_slope:.4f}, against "
      f"varpi' D varpi / (2 Sbar)^2 = {signed_rate / (2 * UNIFORM_DEPTH) ** 2:.4f}")
print(windows.round(2).to_string())
rates.round(3)

The variance of the change of the distance is affine in the length of the window, with a large
constant, and its slope $r$ is the rate; the two parts of the statement add up to it within the
noise of thirty paths.  The geometric mid moves by $\gamma$ a unit of signed count, well below the
idealised $1/(2\bar S)$, so the aligned part is of the order of the idealised mid's own rate.  At
$\eta = 0.2$ it is most of $r$; at $\eta = 0.08$ the geometric mid moves more besides, and
$\sigma_\perp^2$, the motion the signed count does not explain, is the larger part.  The rate falls
as $\eta$ rises.  These are estimates over
windows of at most 150 s; the slope $\gamma$ estimated over a window keeps rising as the window
lengthens, and settles only over windows of the order of an hour, beyond this notebook.

The statement needs the second hypothesis as well, that no side of the geometric book empties.
When one does, the next limit order of the other direction is priced against the fixed reference
price of Definition 1.4.8 (the footnote), not against the book, and the mid jumps to wherever the
reference price puts it.  That fallback is not translation invariant: a mid that has wandered from
the reference price is pulled back towards it.  The simulator is left as it is; we measure how much
of the motion of the geometric mid such episodes carry, as the share of its quadratic variation,
the sum of its squared moves after 150 s, made across an empty side:

In [ ]:
EPISODE_DECAYS = (1.0, 0.5, 0.2, 0.08)
episodes = []
for decay in EPISODE_DECAYS:
    moves = pd.concat([
        geometric_mid_moves(path, with_decay(LOTS, decay), UNIFORM_DEPTH, OPENING, WIDE, m)
        for m, path in enumerate(session_paths)
    ])
    moves = moves[moves["TimeStamp"] > SETTLED]
    squared = moves["Move"] ** 2
    episodes.append({
        "η": decay, "moves of the mid": len(moves),
        "across an empty side": int(moves["AcrossEmptySide"].sum()),
        "share of the quadratic variation":
            squared[moves["AcrossEmptySide"]].sum() / squared.sum(),
    })
pd.DataFrame(episodes).set_index("η").round(3)

At $\eta = 1$ and $\eta = 0.5$ the geometric book concentrates its size on few levels, its sides
empty, and the episodes carry half and nearly all of the quadratic variation of its mid: there the
motion of the mid is set by the reference price, not by the placement.  At $\eta = 0.2$ and $0.08$ no side emptied in these
sessions, and the rates above are the geometric book's own.

The change of the distance from its value at 150 s, over the paths, against the line $c + r h$:

In [ ]:
settled = int(SETTLED / 5.0)
after = SESSION_READ[settled:] - SETTLED
departures, departure_curves = {"sizes in lots": {}}, {"sizes in lots": {}}
for decay in SESSION_DECAYS:
    distance = session_mid[decay] - session_idealised
    change = distance[:, settled:] - distance[:, [settled]]
    label = f"η = {decay:g}"
    departures["sizes in lots"][label] = np.abs(change)
    departure_curves["sizes in lots"][label] = {
        "√E[(Δ distance)²]": root_mean_square(change),
        "√(c + r h), the least-squares line": np.sqrt(
            rates.loc[decay, "c"] + rates.loc[decay, "r"] * after),
    }
fan_panels_figure(departures, after, departure_curves, SESSION_PATHS, "ticks", True,
                  title="|Δ(P<sup>m,geo</sup> − P<sup>m,id</sup>)| after 150 s, withdrawals on, "
                        "30 paths")

## 7. The two regimes

The numbers of the two regimes side by side, from the fans above: the median first discrepancy with
orders of one share, the rate at which the variance of the distance grows, and $2\bar S\gamma$.
Without withdrawals the last two are read at 300 s with unit sizes, and describe that horizon, not
a limit; with withdrawals they are the slopes of section 6.

In [ ]:
medians = pd.DataFrame(summary).set_index(["book", "η"])["median χ≠ (s)"]
short_medians = pd.DataFrame(short_summary).set_index("η")["median χ≠ (s)"]
comparison = pd.DataFrame(
    [
        {"regime": "no withdrawals", "η": decay,
         "median χ≠ (s)": medians["S̄ = 5", decay],
         "variance rate (ticks²/s)": moments.loc[("unit sizes", decay, 300), "Var(distance) / t"],
         "2 Sbar gamma": moments.loc[("unit sizes", decay, 300), "2 Sbar gamma_t"]}
        for decay in DISTANCE_DECAYS
    ] + [
        {"regime": "withdrawals", "η": decay,
         "median χ≠ (s)": short_medians[decay],
         "variance rate (ticks²/s)": rates.loc[decay, "r"],
         "2 Sbar gamma": rates.loc[decay, "2 Sbar gamma"]}
        for decay in SESSION_DECAYS
    ]
).set_index(["regime", "η"])
comparison.round(3)

Each regime shows what the other cannot.

Without withdrawals the two placements send the same orders for a while, and the first discrepancy
is a statement about the flow: its law is the averaged product of section 4, and at $\bar S = 50$
the Riccati value.  But the geometric book has no stationary depth there.  Limit orders and market
orders arrive at one rate, its depth is a walk, and the slope $\gamma_t$ and the rate of the
distance drift with the length of the session.

With withdrawals the books part within a fraction of a second, so the first discrepancy says
little.  With the sizes the specification was calibrated for, the geometric book is stationary,
and past a constant the variance of the distance grows at a constant rate, set by the effective
impact $\gamma(\eta)$ against the idealised $1/(2\bar S)$ and by the variance the signed count does
not explain.  That holds where no side of the geometric book empties; at large $\eta$ the episodes
priced against the reference price carry most of the motion of its mid.

The withdrawal-free flow is balanced on purpose.  Deleting the withdrawal coordinates from the
resilient kernel without rescaling it leaves limit orders arriving faster than the market orders
that consume their side, by about a tenth of a share a second at unit sizes.  The geometric book
then gains that depth on each side without bound, its best quotes are cleared ever more rarely, and
at $\eta = 1$ its mid-price comes to a halt.